# Week 2 Day 3 — Conversational AI / Chatbot (Anthropic edition)

> **Targets Gradio 6.** Two Gradio-6 changes are handled here: (1) `type="messages"` was removed (messages format is now the only option), so we don't pass it; (2) history `content` is now a **list of blocks**, not a string, so we flatten it with a `to_text()` helper before handing it to Anthropic (which wants a plain string). The helper works on older Gradio too.

In [15]:
import os
from dotenv import load_dotenv
import anthropic
import gradio as gr

load_dotenv(override=True)
print("Anthropic key set" if os.getenv("ANTHROPIC_API_KEY") else "Anthropic key NOT set")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

Anthropic key set


In [16]:
def to_text(content):
    """Gradio 6 gives message content as a list of blocks; older Gradio gave a plain string.
    Anthropic wants a plain string here, so flatten it. Handles both versions safely."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    return str(content)

In [17]:
system_message = "You are a helpful assistant"

## The chat callback

We rebuild each history item as a clean `{"role", "content": <string>}` via `to_text`, append the new user turn, and stream. System stays separate.

In [18]:
def chat(message, history):
    messages = [{"role": h["role"], "content": to_text(h["content"])} for h in history]
    messages.append({"role": "user", "content": message})
    with client.messages.stream(
        model=MODEL, max_tokens=1000,
        system=system_message,           # system is SEPARATE, never in the list
        messages=messages,
    ) as stream:
        response = ""
        for text in stream.text_stream:
            response += text
            yield response

In [ ]:
gr.ChatInterface(fn=chat).launch()   # no type="messages" in Gradio 6

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Context and one-shot prompting via the system message

In [20]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.' \
Encourage the customer to buy hats if they are unsure what to get."

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 

In [22]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 

## Dynamically adjusting context

Modify the system message based on the user's message before sending.

In [ ]:
def chat(message, history):
    relevant_system_message = system_message
    if "belt" in message.lower():
        relevant_system_message += " The store does not sell belts; if asked for belts, point out other items on sale."

    messages = [{"role": h["role"], "content": to_text(h["content"])} for h in history]
    messages.append({"role": "user", "content": message})
    with client.messages.stream(
        model=MODEL, max_tokens=1000,
        system=relevant_system_message,
        messages=messages,
    ) as stream:
        response = ""
        for text in stream.text_stream:
            response += text
            yield response

In [ ]:
gr.ChatInterface(fn=chat).launch()